In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("Food_Delivery_Times.csv")
df.head()

In [ ]:
df.drop(columns=["Order_ID"],inplace=True,axis=1)

In [ ]:
df.columns.tolist()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum(axis=1).value_counts().sort_index()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Weather"].value_counts(dropna=False)

In [ ]:
df["Traffic_Level"].value_counts(dropna=False)

In [ ]:
df["Time_of_Day"].value_counts(dropna=False)

In [ ]:
df["Vehicle_Type"].value_counts(dropna=False)

In [ ]:
categorical=df.select_dtypes(include=["object"]).columns
categorical

In [ ]:
df.head()

In [ ]:
df["Delivery_Time_min"].hist(figsize=(4,4),bins=30)
plt.tight_layout()
plt.show()

In [ ]:
df["Delivery_Time_min"].describe()

In [ ]:
df["Delivery_Time_min"].sort_values(ascending=False).head(10)

In [ ]:
df.loc[df["Delivery_Time_min"].idxmax()]

In [ ]:
X=df.drop(columns=["Delivery_Time_min"])
y=df["Delivery_Time_min"]

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [ ]:
numerical=["Distance_km","Preparation_Time_min","Courier_Experience_yrs"]

In [ ]:
num_pipe=Pipeline([("imputer",SimpleImputer(strategy="median"))])
cat_pipe=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown='ignore'))])

In [ ]:
preprocessor=ColumnTransformer([("num",num_pipe,numerical),("cat",cat_pipe,categorical)])

In [ ]:
X_train_pro=preprocessor.fit_transform(X_train)
X_test_pro=preprocessor.transform(X_test)

In [ ]:
X_train_pro.shape,X_test_pro.shape

In [ ]:
feature_names=preprocessor.get_feature_names_out()
feature_names

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
models={
    "LinearRegression":LinearRegression(),
    "DecisionTree":DecisionTreeRegressor(random_state=42),
    "Random Forest":RandomForestRegressor(n_estimators=100,random_state=42),
    "Gradient Boosting":GradientBoostingRegressor(random_state=42),
    "XGBoost":XGBRegressor(n_estimators=100,random_state=42)
}

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

results = []

for name, model in models.items():

    model.fit(X_train_pro, y_train)
    y_pred = model.predict(X_test_pro)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:

error_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

error_df["Error"] = error_df["Actual"] - error_df["Predicted"]
error_df["Absolute_Error"] = abs(error_df["Error"])

error_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

lr_pred = models["LinearRegression"].predict(X_test_pro)

plt.figure(figsize=(7, 5))
plt.scatter(y_test, lr_pred)

plt.xlabel("Actual Delivery Time")
plt.ylabel("Predicted Delivery Time")
plt.title("Actual vs Predicted - Linear Regression")

plt.show()

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rf_grid.fit(X_train_pro, y_train)


print("Best Parameters:")
print(rf_grid.best_params_)


print("\nBest CV RMSE:")
print(-rf_grid.best_score_)


best_rf = rf_grid.best_estimator_

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_pred = best_rf.predict(X_test_pro)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Tuned Random Forest")
print("MAE :", rf_mae)
print("MSE :", rf_mse)
print("RMSE:", rf_rmse)
print("R2  :", rf_r2)

# Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV


gb = GradientBoostingRegressor(random_state=42)


gb_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3, 5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}


gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=gb_param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

gb_grid.fit(X_train_pro, y_train)


print("Best Parameters:")
print(gb_grid.best_params_)


print("\nBest CV RMSE:")
print(-gb_grid.best_score_)

# Best model
best_gb = gb_grid.best_estimator_

In [ ]:
gb_pred = best_gb.predict(X_test_pro)

gb_mae = mean_absolute_error(y_test, gb_pred)
gb_mse = mean_squared_error(y_test, gb_pred)
gb_rmse = np.sqrt(gb_mse)
gb_r2 = r2_score(y_test, gb_pred)

print("Tuned Gradient Boosting")
print("MAE :", gb_mae)
print("MSE :", gb_mse)
print("RMSE:", gb_rmse)
print("R2  :", gb_r2)

# XGBOOST REGRESSOR

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

xgb = XGBRegressor(
    random_state=42
)

xgb_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

xgb_grid.fit(X_train_pro, y_train)

print("Best Parameters:")
print(xgb_grid.best_params_)

print("\nBest CV RMSE:")
print(-xgb_grid.best_score_)

best_xgb = xgb_grid.best_estimator_

In [ ]:
xgb_pred = best_xgb.predict(X_test_pro)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_pred)

print("Tuned XGBoost")
print("MAE :", xgb_mae)
print("MSE :", xgb_mse)
print("RMSE:", xgb_rmse)
print("R2  :", xgb_r2)

In [ ]:
final_results = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "MAE": 6.063415,
        "MSE": 80.141359,
        "RMSE": 8.952171,
        "R2": 0.821204
    },
    {
        "Model": "Tuned Random Forest",
        "MAE": rf_mae,
        "MSE": rf_mse,
        "RMSE": rf_rmse,
        "R2": rf_r2
    },
    {
        "Model": "Tuned Gradient Boosting",
        "MAE": gb_mae,
        "MSE": gb_mse,
        "RMSE": gb_rmse,
        "R2": gb_r2
    },
    {
        "Model": "Tuned XGBoost",
        "MAE": xgb_mae,
        "MSE": xgb_mse,
        "RMSE": xgb_rmse,
        "R2": xgb_r2
    }
])

final_results

In [ ]:
final_model = LinearRegression()
final_model.fit(X_train_pro, y_train)

In [ ]:
import joblib

joblib.dump(final_model, "final_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("Model and preprocessor saved successfully!")